# Integrate apparent magnitude estimation

- choose one aperture size
- integrate fluxes across time series and combine errors
    - use weighted contributions to flux?
- calculate zero point from gaia sources
    - try with comparison stars
    - try with all sources
    - try with weighted sources


In [48]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from astropy.table import Table, join

## Load the tables

Point `reduce_dir` at a folder written by `pw-aperture` (i.e. containing `sources.*`,
`images.*`, `flux.*`).

In [49]:
#reduce_dir = Path("/home/adamarthurryan/Desktop/baron obs/pandora/pandorahatp11multibandseries_20260701_baron_gp/reduce")
#reduce_dir = Path("/home/adamarthurryan/Desktop/baron obs/mkr421/mrk421microvariabilityirg_20260708_baron_ip/reduce")
reduce_dir = Path("/home/adamarthurryan/Desktop/baron obs/pandora/pandorahatp11multibandseries_20260708_baron_rp/reduce")
gaia_filter_key = "r_sdss_mag"

source_table = Table.read(next(reduce_dir.glob("sources.*")))
image_table = Table.read(next(reduce_dir.glob("images.*")))
flux_table = Table.read(next(reduce_dir.glob("flux.*")))


len(source_table), len(image_table), len(flux_table)

(100, 13, 1300)

## Aggregate measurements

Source flux, background flux, and source flux errors are aggregated across all images.

In [50]:
from functools import partial

# group by image
grouped = flux_table.group_by("source_id")

#define aggregate functions
sum = partial(np.sum, axis=0)
sum_in_quadrature = lambda x: np.sqrt(np.sum(x**2, axis=0))
 
# aggregate measurements to new table
table = Table([grouped.groups.keys["source_id"]])
table["src_flux"] = grouped["src_flux"].groups.aggregate(sum)
table["bkg_flux"] = grouped["bkg_flux"].groups.aggregate(sum)
table["src_flux_error"] = grouped["src_flux_error"].groups.aggregate(sum_in_quadrature)



## Pick an aperture

for now, just pick a single aperture 

In [51]:
aperture_index = 20
table_1ap = table.copy()

table_1ap["src_flux"] = table["src_flux"][:, aperture_index]
table_1ap["src_flux_error"] = table["src_flux_error"][:, aperture_index]
table_1ap["bkg_flux"] = table["bkg_flux"][:, aperture_index]

table_1ap

source_id,src_flux,bkg_flux,src_flux_error
int64,float64,float64,float64
0,16998285.105780624,2366871.0947855697,4146.687554938175
1,15430303.858248103,2366101.5158244963,3954.2950016279733
2,10094017.932624755,2379038.3245583116,3488.9961328010636
3,8687388.977532905,2358086.290587308,2979.9951563833465
4,4556955.039708437,2347204.069022002,2166.4760740908932
5,4239204.943789786,2350362.369471842,2093.079842017788
6,3433603.2410630593,2346095.4139746595,1889.2157514434668
7,2842624.33441196,2346465.243285456,1725.7473795394028
8,2638123.099275562,2343767.477944376,1663.082699977807


## Global zero point from Gaia-matched calibrators

In [52]:

table_1ap["inst_mag"] = -2.5 * np.log10(table_1ap["src_flux"])

source_flux_table = join(table_1ap, source_table)
source_flux_table = source_flux_table[source_flux_table["gaia_matched"]]

zero_points = source_flux_table[gaia_filter_key] - source_flux_table["inst_mag"]
zero_point = np.median(zero_points)

source_flux_table["apparent_mag"] = source_flux_table["inst_mag"]+zero_point
display(source_flux_table["source_id", "src_flux", "src_flux_error", "apparent_mag", gaia_filter_key])



/home/adamarthurryan/micromamba/envs/photometry/lib/python3.14/site-packages/numpy/_core/fromnumeric.py:840: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedColumn.
  a.partition(kth, axis=axis, kind=kind, order=order)


source_id,src_flux,src_flux_error,apparent_mag,r_sdss_mag
,,,,mag
int64,float64,float64,float64,float32
1,15430303.858248103,3954.2950016279733,9.175971480004765,9.175393
3,8687388.977532905,2979.9951563833465,9.799684507350687,9.80064
4,4556955.039708437,2166.4760740908932,10.500220816713124,10.523803
5,4239204.943789786,2093.079842017788,10.578696643562125,10.572748
6,3433603.2410630593,1889.2157514434668,10.807532400453137,10.812952
7,2842624.33441196,1725.7473795394028,11.01260900223679,11.004954
8,2638123.099275562,1663.082699977807,11.093670034530174,11.100838
9,2557731.500863452,1647.6329236987142,11.127270295381184,11.124822


## Comparison sequence from AAVSO VSP

Fetch a standard photometric comparison-star sequence for `target_star_name` from the
AAVSO Variable Star Plotter, then cross-match it against `source_table` to find the
`source_id` of the target star and of each comparison star.

In [53]:
from photometry_workflow.common.aavso import fetch_vsp_sequence

target_star_name = "HAT-P-11"
fov = 60  # arcmin - should roughly match the imaged field so VSP doesn't return comp
          # stars outside our footprint (check against source_table's ra/dec spread)

target_coord, sequence_table = fetch_vsp_sequence(target_star_name, fov=fov)

print(target_coord)
sequence_table

<SkyCoord (ICRS): (ra, dec) in deg
    (297.709375, 48.08086111)>


auid,label,ra,dec,B_mag,B_error,V_mag,V_error,Rc_mag,Rc_error,Ic_mag,Ic_error
,,deg,deg,,,,,,,,
str11,str3,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
000-BNV-365,94,297.37699999999995,48.126472222222226,10.307,0.007,9.357,0.003,8.76,0.006,8.204,0.008
000-BNV-366,106,297.87287499999996,47.88908333333333,10.982,0.013,10.6,0.007,10.263,0.015,9.946,0.02
000-BNV-367,108,297.5586666666666,48.19883333333333,nan,nan,10.75,0.008,10.395,0.015,10.061,0.02
000-BLJ-489,120,298.36362499999996,47.837694444444445,12.578,0.197,12.007,0.032,11.611,0.204,11.239,0.286
000-BLJ-490,138,298.41933333333327,47.83486111111112,14.609,0.253,13.835,0.016,13.369,0.186,12.933,0.262
000-BLJ-491,140,298.32270833333325,47.788666666666664,15.026,0.219,13.979,0.025,13.28,0.127,12.631,0.179
000-BLJ-492,142,298.3951666666666,47.76475,14.8,0.218,14.174,0.049,13.661,0.109,13.182,0.146
000-BLJ-493,144,298.4489166666666,47.86208333333334,14.939,0.195,14.355,0.028,14.017,0.135,13.699,0.189


VSP's target coordinate is given at a fixed catalog epoch, not corrected for proper
motion to the time of observation - this matters for HAT-P-11, which has high enough
proper motion to be off by several arcsec over the years since. Correct it using
Gaia DR3's astrometric solution before matching against `source_table`.

In [54]:
from astropy.time import Time
import astropy.units as u
from photometry_workflow.common.cross_match import propagate_gaia_position

obstime = Time(np.median(image_table["time"]), format="jd")
target_coord_uncorrected = target_coord
target_coord = propagate_gaia_position(target_coord_uncorrected, obstime)

print(f"observation epoch: {obstime.jyear:.2f}")
print(f"VSP target coord:              {target_coord_uncorrected.to_string('hmsdms')}")
print(f"proper-motion corrected coord: {target_coord.to_string('hmsdms')}")
print(f"shift: {target_coord.separation(target_coord_uncorrected).to(u.arcsec)}")

observation epoch: 2026.52
VSP target coord:              19h50m50.25s +48d04m51.1s
proper-motion corrected coord: 19h50m50.58096663s +48d04m57.27273425s
shift: 7.007332130692249 arcsec


In [55]:
from astropy.coordinates import SkyCoord, match_coordinates_sky
import astropy.units as u

min_accuracy = 2 * u.arcsec

source_coords = SkyCoord(source_table["centroid_ra"], source_table["centroid_dec"])

# match the target star itself. VSP's own target position isn't always precise enough
# (e.g. proper motion since the catalog epoch, or a poor centroid on a bright/saturated
# core), so don't hard-fail on a distant match - just flag it for a sanity check.
target_idx, target_sep, _ = match_coordinates_sky(target_coord, source_coords)
target_source_id = source_table["source_id"][target_idx]
if target_sep > min_accuracy:
    print(f"warning: nearest source to target star {target_star_name!r} is {target_sep.to(u.arcsec)} away (source_id={target_source_id}) - verify this is really the target")

# match each comparison star in the sequence
sequence_coords = SkyCoord(sequence_table["ra"], sequence_table["dec"])
comp_idx, comp_sep, _ = match_coordinates_sky(sequence_coords, source_coords)
sequence_table["source_id"] = source_table["source_id"][comp_idx]
sequence_table["match_sep"] = comp_sep.to(u.arcsec)
sequence_table["matched"] = comp_sep < min_accuracy

comparison_source_ids = sequence_table["source_id"][sequence_table["matched"]]

print(f"target_source_id = {target_source_id}")
print(f"{sequence_table['matched'].sum()} of {len(sequence_table)} comparison stars matched a detected source")

matched_table = sequence_table["auid", "label", "source_id"][sequence_table['matched']]
matched_table
#sequence_table["auid", "label", "source_id", "match_sep", "matched", "V_mag", "B_mag"]


target_source_id = 0
2 of 10 comparison stars matched a detected source


auid,label,source_id
str11,str3,int64
000-BNV-365,94,1
000-BNV-366,106,5


In [57]:
# do photometry with the comparison sequence

table = join(table_1ap, matched_table, keys=["source_id"])
table = join(table, source_table, keys=["source_id"])
display(table)

comp_idx = 1
check_idx = 0

zero_points = table[gaia_filter_key] - table["inst_mag"]
zero_point = zero_points[comp_idx]

check_mag = table["inst_mag"][check_idx]+zero_point
check_mag_expected = table[gaia_filter_key][check_idx]
target_mag = table_1ap["inst_mag"][target_idx]+zero_point
target_mag_expeted = source_flux_table[gaia_filter_key][target_idx]

print("zero point: ", zero_point)
print("check mag: ", check_mag)
print("check mag expected: ", check_mag_expected)
print("target mag: ", target_mag)
print("target mag gaia: ", target_mag_expeted)



source_id,src_flux,bkg_flux,src_flux_error,inst_mag,auid,label,centroid_x,centroid_y,centroid_ra,centroid_dec,gaia_match_err,gaia_matched,ra,dec,gaia_source_id,c_star,u_jkc_flux,u_jkc_flux_error,u_jkc_mag,u_jkc_flag,b_jkc_flux,b_jkc_flux_error,b_jkc_mag,b_jkc_flag,v_jkc_flux,v_jkc_flux_error,v_jkc_mag,v_jkc_flag,r_jkc_flux,r_jkc_flux_error,r_jkc_mag,r_jkc_flag,i_jkc_flux,i_jkc_flux_error,i_jkc_mag,i_jkc_flag,u_sdss_flux,u_sdss_flux_error,u_sdss_mag,u_sdss_flag,g_sdss_flux,g_sdss_flux_error,g_sdss_mag,g_sdss_flag,r_sdss_flux,r_sdss_flux_error,r_sdss_mag,r_sdss_flag,i_sdss_flux,i_sdss_flux_error,i_sdss_mag,i_sdss_flag,z_sdss_flux,z_sdss_flux_error,z_sdss_mag,z_sdss_flag,y_ps1_flux,y_ps1_flux_error,y_ps1_mag,y_ps1_flag,f606w_acswfc_flux,f606w_acswfc_flux_error,f606w_acswfc_mag,f606w_acswfc_flag,f814w_acswfc_flux,f814w_acswfc_flux_error,f814w_acswfc_mag,f814w_acswfc_flag
,,,,,,,,,deg,deg,deg,,deg,deg,,,W / (nm m2),W / (nm m2),mag,,W / (nm m2),W / (nm m2),mag,,W / (nm m2),W / (nm m2),mag,,W / (nm m2),W / (nm m2),mag,,W / (nm m2),W / (nm m2),mag,,W / (nm m2),W / (nm m2),mag,,W / (nm m2),W / (nm m2),mag,,W / (nm m2),W / (nm m2),mag,,W / (nm m2),W / (nm m2),mag,,W / (nm m2),W / (nm m2),mag,,W / (nm m2),W / (nm m2),mag,,W / (nm m2),W / (nm m2),mag,,W / (nm m2),W / (nm m2),mag,
int64,float64,float64,float64,float64,str11,str3,float64,float64,float64,float64,float64,bool,float64,float64,int64,float32,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16,float32,float32,float32,int16
1,15430303.858248103,2366101.5158244963,3954.2950016279733,-17.970936196016766,000-BNV-365,94,2819.494658119658,2474.8824786324785,297.37706145133086,48.126283053886716,0.00020678315512549433,True,297.3769597811138,48.1264783837966,2086526830740438784,-0.000592947,1.6202404e-15,4.888863e-17,11.1271515,0,4.2995604e-15,1.3546876e-17,10.44544,0,5.9295673e-15,1.2411582e-17,9.481342,0,5.838461e-15,8.549074e-18,8.957454,0,4.8628784e-15,6.329682e-18,8.461567,0,8.233836e-28,2.3887867e-29,11.923494,0,3.8690453e-27,1.01454145e-29,9.910291,0,7.703407e-27,1.341977e-29,9.175393,0,9.846617e-27,1.2676513e-29,8.897482,0,1.1106337e-26,1.807117e-29,8.762473,0,1.14642005e-26,2.5947976e-29,8.758341,0,5.9286432e-15,7.996289e-18,9.229511,0,4.7884793e-15,6.1134177e-18,8.464306,0
5,4239204.943789786,2350362.369471842,2093.079842017788,-16.568211032459406,000-BNV-366,106,1010.3913043478261,376.1217391304348,297.87311312898316,47.88889499291829,0.00018035956324294442,True,297.87286703426696,47.88896777165046,2086504698763807360,-0.0024727583,1.735401e-15,2.1315453e-17,11.0526,0,2.3602507e-15,8.15327e-18,11.096604,0,1.9753283e-15,4.3759033e-18,10.674802,0,1.5250663e-15,2.4248084e-18,10.414978,0,1.0179146e-15,1.0851394e-18,10.159521,0,8.731151e-28,1.04287346e-29,11.859821,0,1.7107005e-27,4.283214e-30,10.796365,0,2.1268697e-27,3.9594415e-30,10.572748,0,2.1845361e-27,2.3858247e-30,10.532302,0,2.1074448e-27,2.6993597e-30,10.56701,0,2.08231e-27,3.920224e-30,10.610336,0,1.7541971e-15,2.3602379e-18,10.551704,1,9.993517e-16,1.0060644e-18,10.165504,1


zero point:  27.140959216663507
check mag:  9.170023020646742
check mag expected:  9.175393
target mag:  9.064946443609553
target mag gaia:  9.175393
